In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print("ANSI mode:", spark.conf.get("spark.sql.ansi.enabled"))

def latest_version(table):
    """Newest _feed_version in a bronze table. YYYY-MM-DD strings sort correctly."""
    return spark.table(table).agg(F.max("_feed_version")).collect()[0][0]

def clean_strings(df):
    """Trim every source string column; turn empty or whitespace-only into true null.
    Metadata columns (leading underscore) are left alone."""
    out = []
    for f in df.schema.fields:
        if f.name.startswith("_") or f.dataType.simpleString() != "string":
            out.append(F.col(f.name))
        else:
            t = F.trim(F.col(f.name))
            out.append(F.when(t == "", None).otherwise(t).alias(f.name))
    return df.select(out)

def cast_columns(df, spec):
    """Cast columns per {name: SQL type} using try_cast, and count failures.
    A failure = the source had a value, but the cast produced null.
    Columns in the spec but absent from the data are skipped and reported."""
    present = {c: t for c, t in spec.items() if c in df.columns}
    missing = sorted(set(spec) - set(present))
    casted = df.select("*", *[F.expr(f"try_cast(`{c}` AS {t})").alias(f"__c_{c}")
                              for c, t in present.items()])
    fails = casted.select([
        F.sum((F.col(c).isNotNull() & F.col(f"__c_{c}").isNull()).cast("int")).alias(c)
        for c in present]).collect()[0].asDict()
    out = casted.select([F.col(f"__c_{c}").alias(c) if c in present else F.col(c)
                         for c in df.columns])
    return out, {c: int(v or 0) for c, v in fails.items()}, missing

def guard_unique(df, keys):
    """Keep one row per natural key. Returns (df, rows_removed)."""
    w = Window.partitionBy(*keys).orderBy(F.desc("_ingested_at"))
    out = df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
    return out, df.count() - out.count()

def write_silver(df, name):
    (df.withColumn("_silver_loaded_at", F.current_timestamp())
       .write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"transit.silver.{name}"))
    print(f"transit.silver.{name:8} {spark.table(f'transit.silver.{name}').count():>10,} rows")

print("helpers ready")

In [0]:
BRONZE = {t: f"transit.bronze.gtfs_{t}" for t in ["stops", "routes", "trips"]}

versions = {t: latest_version(tbl) for t, tbl in BRONZE.items()}
print("latest version per table:", versions)

assert len(set(versions.values())) == 1, f"tables disagree on latest version: {versions}"
FEED_VERSION = versions["stops"]

def bronze(t):
    return spark.table(BRONZE[t]).filter(F.col("_feed_version") == FEED_VERSION)

print(f"\nbuilding silver from feed_version {FEED_VERSION}")
for t in BRONZE:
    print(f"  bronze {t:6} {bronze(t).count():>10,} rows")

In [0]:
STOP_TYPES = {
    "stop_lat":            "DOUBLE",
    "stop_lon":            "DOUBLE",
    "location_type":       "INT",
    "wheelchair_boarding": "INT",
    "vehicle_type":        "INT",
}

LOCATION_TYPES = spark.createDataFrame(
    [(0, "Stop / Platform"), (1, "Station"), (2, "Entrance / Exit"),
     (3, "Generic Node"),    (4, "Boarding Area")],
    "location_type int, location_type_name string")

stops_raw = bronze("stops")
stops, stop_fails, stop_missing = cast_columns(clean_strings(stops_raw), STOP_TYPES)

print("cast failures:", stop_fails)
print("spec columns not in data:", stop_missing or "none")
assert sum(stop_fails.values()) == 0, f"values that would have silently become null: {stop_fails}"

# GTFS: an empty location_type means 0 (stop/platform). Make that explicit.
stops = (stops
    .withColumn("location_type", F.coalesce("location_type", F.lit(0)))
    .join(F.broadcast(LOCATION_TYPES), "location_type", "left"))

stops, removed = guard_unique(stops, ["stop_id"])
print("duplicate stop_ids removed:", removed)

write_silver(stops, "stops")

In [0]:
s = spark.table("transit.silver.stops")

# Null coordinates, explained by type
(s.groupBy("location_type", "location_type_name")
   .agg(F.count("*").alias("stops"),
        F.sum(F.col("stop_lat").isNull().cast("int")).alias("null_lat"))
   .orderBy("location_type").display())

# Anything outside the MBTA region? (Commuter rail reaches Providence and Worcester.)
outside = s.filter(~F.col("stop_lat").between(41.0, 43.5) | ~F.col("stop_lon").between(-72.5, -69.5))
print("stops outside the service-area box:", outside.count())
outside.select("stop_id", "stop_name", "stop_lat", "stop_lon", "location_type_name").limit(20).display()

# Every parent_station should be a real stop
orphans = (s.filter(F.col("parent_station").isNotNull())
             .join(s.select(F.col("stop_id").alias("parent_station")), "parent_station", "left_anti"))
print("stops whose parent_station doesn't exist:", orphans.count())

In [0]:
ROUTE_TYPES_SPEC = {"route_type": "INT", "route_sort_order": "INT", "listed_route": "INT"}

ROUTE_TYPE_NAMES = spark.createDataFrame(
    [(0, "Light Rail"), (1, "Subway"), (2, "Rail"), (3, "Bus"), (4, "Ferry"),
     (5, "Cable Tram"), (6, "Aerial Lift"), (7, "Funicular"),
     (11, "Trolleybus"), (12, "Monorail")],
    "route_type int, route_type_name string")

routes, route_fails, route_missing = cast_columns(clean_strings(bronze("routes")), ROUTE_TYPES_SPEC)
print("cast failures:", route_fails)
print("spec columns not in data:", route_missing or "none")
assert sum(route_fails.values()) == 0, f"cast failures: {route_fails}"

routes = routes.join(F.broadcast(ROUTE_TYPE_NAMES), "route_type", "left")

unmapped = routes.filter(F.col("route_type_name").isNull()).select("route_type").distinct().collect()
print("route_type values with no name:", [r[0] for r in unmapped] or "none")

routes, removed = guard_unique(routes, ["route_id"])
print("duplicate route_ids removed:", removed)

write_silver(routes, "routes")

spark.table("transit.silver.routes").groupBy("route_type", "route_type_name").count() \
     .orderBy("route_type").display()

In [0]:
TRIP_TYPES = {"direction_id": "INT", "wheelchair_accessible": "INT", "bikes_allowed": "INT"}

trips, trip_fails, trip_missing = cast_columns(clean_strings(bronze("trips")), TRIP_TYPES)
print("cast failures:", trip_fails)
print("spec columns not in data:", trip_missing or "none")
assert sum(trip_fails.values()) == 0, f"cast failures: {trip_fails}"

trips, removed = guard_unique(trips, ["trip_id"])
print("duplicate trip_ids removed:", removed)

# Every trip must belong to a route that exists
orphan_trips = trips.join(spark.table("transit.silver.routes").select("route_id"), "route_id", "left_anti")
n_orphans = orphan_trips.count()
print("trips pointing at a non-existent route:", n_orphans)
assert n_orphans == 0, "trips reference routes missing from this feed version"

write_silver(trips, "trips")

In [0]:
checks = {"stops": "stop_id", "routes": "route_id", "trips": "trip_id"}

for t, key in checks.items():
    silver = spark.table(f"transit.silver.{t}")
    n_silver, n_bronze = silver.count(), bronze(t).count()
    dupes = silver.groupBy(key).count().filter("count > 1").count()
    print(f"{t:6}  bronze {n_bronze:>8,}  silver {n_silver:>8,}  duplicate keys {dupes}")
    assert n_silver == n_bronze, f"{t}: rows lost or gained between bronze and silver"
    assert dupes == 0, f"{t}: duplicate natural keys"

print("\nsilver dimensions OK for feed_version", FEED_VERSION)

In [0]:
all_versions = sorted(r[0] for r in spark.table(BRONZE["stops"])
                      .select("_feed_version").distinct().collect())
prev, curr = all_versions[-2], all_versions[-1]
print(f"comparing stops: {prev} -> {curr}")

def version(v):
    return (clean_strings(spark.table(BRONZE["stops"]).filter(F.col("_feed_version") == v))
            .select("stop_id", "stop_name", "stop_lat", "stop_lon", "wheelchair_boarding"))

a, b = version(prev), version(curr)

added   = b.join(a, "stop_id", "left_anti")
removed = a.join(b, "stop_id", "left_anti")
changed = (a.alias("a").join(b.alias("b"), "stop_id")
             .filter(~F.expr("""a.stop_name <=> b.stop_name AND a.stop_lat <=> b.stop_lat
                              AND a.stop_lon <=> b.stop_lon
                              AND a.wheelchair_boarding <=> b.wheelchair_boarding""")))

print(f"added {added.count()} · removed {removed.count()} · changed {changed.count()}")
changed.select("stop_id", "a.stop_name", "b.stop_name", "a.stop_lat", "b.stop_lat").limit(20).display()